# Demand Forecasting with Price Sensitivity - Part 2

This notebook builds a demand forecasting model that incorporates price sensitivity to enable revenue optimization.

Part 2: Model Training

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib.pyplot as plt
import seaborn as sns
from calplot import calplot as clp
import mplcyberpunk
plt.style.use("cyberpunk")

from catboost import CatBoostRegressor

from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import gc
import holidays

pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [2]:
# Load preprocessed data from part 1
df = pd.read_pickle('df_processed_part1.pkl')
df_test = pd.read_pickle('df_test_processed_part1.pkl')
print("Data loaded from part 1")

Data loaded from part 1


## Model Training with Revenue Objective

In [3]:
# Подготовка фичей для моделирования
# FIX: Save price_base values before dropping them
price_base_values = df["price_base"].copy()  # Save for later use

cols_drop = ["date", "day", "dayofweek", "week", "month"]  # FIX: Remove 'price_base' from cols_drop

# Удаляем sum_total если есть (не нужен в фичах)
if "sum_total" in df.columns:
    cols_drop.append("sum_total")

df_model = df.drop(columns=cols_drop)

# Фичи и таргеты
X = df_model.drop("quantity", axis=1)
y_quantity = df_model["quantity"]

# Debug: Check if sum_total exists
print(f"DEBUG: df_model columns: {list(df_model.columns)}")
print(f"DEBUG: 'sum_total' in df_model.columns: {'sum_total' in df_model.columns}")
if "sum_total" in df_model.columns:
    print(f"DEBUG: sum_total stats - mean: {df_model['sum_total'].mean():.2f}, std: {df_model['sum_total'].std():.2f}")

# Revenue таргет: приоритет sum_total, fallback quantity * price_base
if "sum_total" in df_model.columns:
    y_revenue = df_model["sum_total"]
    print("Note: Using sum_total as revenue target")
else:
    y_revenue = df["quantity"] * price_base_values  # Use saved price_base values
    print("Note: Recreated revenue target from quantity * price_base")

# Категориальные фичи → string
categorical_cols = X.select_dtypes("object").columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype(str)

print(f"Features shape: {X.shape}")
print(f"Quantity target shape: {y_quantity.shape}")
print(f"Revenue target shape: {y_revenue.shape}")
print(f"DEBUG: Revenue target stats - mean: {y_revenue.mean():.2f}, std: {y_revenue.std():.2f}")

DEBUG: df_model columns: ['item_id', 'quantity', 'price_base', 'store_id', 'price', 'code', 'price_history', 'price_change_from_history', 'price_level', 'promo_day', 'sale_price_before_promo', 'sale_price_time_promo', 'discount_percentage', 'promo_type_code', 'doc_id', 'number_disc_day', 'division', 'format', 'city', 'area', 'year', 'day_sin', 'day_cos', 'dayofweek_sin', 'dayofweek_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_sunday', 'holidays', 'season', 'price_lag_1', 'price_lag_7', 'price_lag_30', 'price_change_1', 'price_change_7', 'price_change_30']
DEBUG: 'sum_total' in df_model.columns: False
Note: Recreated revenue target from quantity * price_base
Features shape: (7671841, 38)
Quantity target shape: (7671841,)
Revenue target shape: (7671841,)
DEBUG: Revenue target stats - mean: 779.75, std: 4773.87


In [4]:
# Split data for training
seed_value = 42
idx = np.arange(len(y_quantity))
idx_train, idx_val = train_test_split(idx, train_size=0.9, random_state=seed_value)

# Prepare training and validation sets
X_train = X.iloc[idx_train]
X_val = X.iloc[idx_val]
y_train_qty = y_quantity.iloc[idx_train]
y_val_qty = y_quantity.iloc[idx_val]
y_train_rev = y_revenue.iloc[idx_train]
y_val_rev = y_revenue.iloc[idx_val]

# Get price information for revenue calculation
price_base_val = price_base_values.iloc[idx_val].values  # Use saved price_base values

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import make_column_transformer

# 1. Числовые колонки: inf → NaN → median (train для обоих наборов)
print("Checking inf/NaN in numerical columns:")
num_cols = X.select_dtypes([np.int32, np.int64, np.float32, np.float64]).columns

for col in num_cols:
    inf_cnt = np.isinf(X_train[col]).sum()
    nan_cnt = np.isnan(X_train[col]).sum()
    
    if inf_cnt > 0 or nan_cnt > 0:
        # inf → NaN для обоих
        for df in [X_train, X_val]:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_val[col] = X_val[col].fillna(med)
        print(f"  '{col}': {inf_cnt} inf, {nan_cnt} NaN → median {med:.4f}")

# 2. Категориальные: str + 'missing' в 1 строку
cat_cols = X.select_dtypes("object").columns.tolist()
print(f"\nCategorical columns ({len(cat_cols)}): {cat_cols}")

for col in cat_cols:
    for df in [X_train, X_val]:
        df[col] = df[col].astype(str).replace('nan', 'missing').fillna('missing')
    print(f"  '{col}': {X_train[col].nunique()} uniques")

# 3. Финальная проверка NaN (редко нужно, но на всякий)
print("\nFinal NaN check...")
for col in X_train.columns:
    if X_train[col].isna().any():
        if col in cat_cols:
            X_train[col].fillna('missing', inplace=True)
            X_val[col].fillna('missing', inplace=True)
        else:
            med = X_train[col].median()
            X_train[col].fillna(med, inplace=True)
            X_val[col].fillna(med, inplace=True)
        print(f"  Fixed '{col}'")

print("✅ All data clean!")

# 4. Memory-efficient preprocessing (LabelEncoder для больших кардинальностей)
print("\nMemory-efficient encoding...")
label_encoders = {}
X_train_enc = X_train.copy()
X_val_enc = X_val.copy()

for col in cat_cols:
    le = LabelEncoder()
    # Fit на объединённых данных (все категории)
    le.fit(pd.concat([X_train[col], X_val[col]]))
    X_train_enc[col] = le.transform(X_train[col])
    X_val_enc[col] = le.transform(X_val[col])
    label_encoders[col] = le
    print(f"  '{col}': {len(le.classes_)} → encoded")

# 5. Scale числовых
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_enc[num_cols])
X_val_num = scaler.transform(X_val_enc[num_cols])

# 6. Объединяем: num_scaled + cat_encoded
X_train_transformed = np.hstack([
    X_train_num, 
    X_train_enc[cat_cols].values
])
X_val_transformed = np.hstack([
    X_val_num, 
    X_val_enc[cat_cols].values
])

print(f"\n✅ Final shapes: train {X_train_transformed.shape}, val {X_val_transformed.shape}")
print(f"Memory: train {X_train_transformed.nbytes/1024**2:.1f}MB")

Checking inf/NaN in numerical columns:


/tmp/ipykernel_3059/1644906002.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].replace([np.inf, -np.inf], np.nan)
/tmp/ipykernel_3059/1644906002.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].fillna(med)
/tmp/ipykernel_3059/1644906002.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/

  'price': 0 inf, 6693809 NaN → median 129.9000
  'code': 0 inf, 6693809 NaN → median 11.0000


/tmp/ipykernel_3059/1644906002.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].fillna(med)
/tmp/ipykernel_3059/1644906002.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val[col] = X_val[col].fillna(med)
/tmp/ipykernel_3059/1644906002.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stabl

  'price_change_from_history': 1491 inf, 0 NaN → median 0.0000
  'promo_type_code': 0 inf, 5535928 NaN → median 5.0000


...

  'division': 2 uniques


/tmp/ipykernel_3059/1644906002.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).replace('nan', 'missing').fillna('missing')
/tmp/ipykernel_3059/1644906002.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).replace('nan', 'missing').fillna('missing')


  'format': 4 uniques


/tmp/ipykernel_3059/1644906002.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).replace('nan', 'missing').fillna('missing')
/tmp/ipykernel_3059/1644906002.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).replace('nan', 'missing').fillna('missing')


  'city': 3 uniques

Final NaN check...
✅ All data clean!

Memory-efficient encoding...
  'item_id': 28309 → encoded
  'doc_id': 11791 → encoded
  'division': 2 → encoded
  'format': 4 → encoded
  'city': 3 → encoded

✅ Final shapes: train (6904656, 33), val (767185, 33)
Memory: train 1738.4MB


In [6]:
# FIX: Create a compatible column_transformer for price scenario functions
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Create a column transformer that matches the manual preprocessing
# For numerical features: apply StandardScaler
# For categorical features: they should already be LabelEncoded, so we pass them as-is (but as floats)
column_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), list(range(len(num_cols)))),
        ('cat', 'passthrough', list(range(len(num_cols), len(num_cols) + len(cat_cols))))
    ],
    remainder='drop'
)

# Fit the column transformer on our manually preprocessed data
# We need to create a mock dataset that matches the structure
# Convert to float to avoid the string conversion issue
mock_train_data = np.hstack([
    X_train_enc[num_cols].values.astype(np.float32), 
    X_train_enc[cat_cols].values.astype(np.float32)
])
column_transformer.fit(mock_train_data)

print("Column transformer created and fitted for price scenario functions")

Column transformer created and fitted for price scenario functions


In [7]:
# Train demand forecasting model
model_path = "demand_forecast_model.cbm"

if os.path.exists(model_path):
    model = CatBoostRegressor()
    model.load_model(model_path)
    print("Loaded existing model")
else:
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.3,
        depth=10,
        l2_leaf_reg=3,
        early_stopping_rounds=50,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_state=seed_value,
        verbose=100
    )

    model.fit(X_train_transformed, y_train_qty, eval_set=[(X_val_transformed, y_val_qty)], verbose=True)
    model.save_model(model_path)
    print("Trained and saved new model")

Loaded existing model


In [9]:
import pickle
import pandas as pd

# Компоненты препроцессинга для сохранения
preprocessing_components = {
    "label_encoders": label_encoders,
    "scaler": scaler,
    "numerical_cols": num_cols,
    "categorical_cols": cat_cols,
    "column_transformer": column_transformer,
    "price_base_values": price_base_values,
}

# Переупорядочиваем X_val: num_cols + cat_cols (для column_transformer)
X_val_reordered = pd.concat([X_val[num_cols], X_val[cat_cols]], axis=1)

# Данные валидации с переупорядоченными фичами
preprocessed_val_data = {
    "X_val_transformed": X_val_transformed,
    "y_val_qty": y_val_qty,
    "y_val_rev": y_val_rev,
    "price_base_val": price_base_val,
    "idx_val": idx_val,
    "X_val_features": X_val_reordered,  # Для price scenario sampling в part 3
}

# Сохраняем
with open("preprocessed_val_data.pkl", "wb") as f:
    pickle.dump(preprocessed_val_data, f)

with open("preprocessing_components.pkl", "wb") as f:
    pickle.dump(preprocessing_components, f)

with open("column_transformer.pkl", "wb") as f:
    pickle.dump(column_transformer, f)

print("✅ Saved:")
print(f"  - preprocessed_val_data.pkl ({len(X_val_reordered)} samples)")
print(f"  - preprocessing_components.pkl")
print(f"  - column_transformer.pkl (backward compatibility)")


✅ Saved:
  - preprocessed_val_data.pkl (767185 samples)
  - preprocessing_components.pkl
  - column_transformer.pkl (backward compatibility)
